# Clean, Source-Organized Copy
Saved outputs and machine-specific metadata are intentionally excluded. The byte-identical original remains in the local audit archive. Prepare the local workbench before running this notebook. Notebooks 01 and 02 execute GPU inference when explicitly enabled; 03 disables its optional GPU activation check by default.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts" / "lab.py").is_file()
)
WORKSPACE = PROJECT_ROOT / ".local" / "workbench"
if not (WORKSPACE / "weight_edit.py").is_file():
    raise RuntimeError("Run python scripts/lab.py prepare --with-local-data first.")
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))

ALLOW_GPU_INFERENCE = False
if not ALLOW_GPU_INFERENCE:
    raise RuntimeError('Set ALLOW_GPU_INFERENCE=True only to deliberately run the GPU demonstration.')


# Replay a published weight-space edit

**Question:** does the published edit change Qwen's thinking behavior on the same small math probes?

We compare base vs edited weights with identical prompts, seeds, and sampling settings. There is no training, optimizer, or gradient calculation. Only two MLP matrices change, temporarily in memory.

Select **Python (weight-space)**. Shut down notebook 01's kernel if you want to free its GPU memory; this notebook loads its own model. All model and edit-data loading here is offline.

This replays the authors' fix; it does **not** independently discover it or reproduce the paper's math benchmark. The default is only two handmade problems with a short generation budget.

Source: https://github.com/RikiyaT/lm_understand-logs/tree/4d09ba6d6a7e891dd8063974dfe81b3d78e3494f/exp8_math/runs/p50c/edit (Rikiya Takehi et al., CC BY 4.0; data used without modification).

In [ ]:
import json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import torch
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration

from download_model import REVISION
from math_probe import PROBLEMS, SAMPLING, run_probes
from weight_edit import UPSTREAM_COMMIT, edited_weights, load_edit, tensor_sha256

if not torch.cuda.is_available():
    raise RuntimeError('Select the Python (weight-space) kernel and check CUDA.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## Keep this small first

Thinking is **on**. Temperature 0.6, top-p 0.95, top-k 20, and generated-token presence penalty 1.5 follow the article's settings. Presence penalty is not the same as repetition penalty; our helper implements the generated-token presence penalty explicitly.

The article uses up to 32,768 tokens and multiple samples per benchmark problem. Here we use at most 512 tokens, one sample, and two toy problems. Hitting this short cap is not proof of a loop, and seeing no gain is not proof the published fix fails.

In [ ]:
PROBE_COUNT = 2
MAX_NEW_TOKENS = 512
SEED = 0
EDIT_LAYERS = (15, 19)

if type(PROBE_COUNT) is not int or not 1 <= PROBE_COUNT <= len(PROBLEMS):
    raise ValueError('Choose between 1 and 4 supplied problems.')
problems = list(PROBLEMS[:PROBE_COUNT])
print('Sampling:', SAMPLING)
for problem in problems:
    print(problem['id'], '-', problem['question'], 'Expected:', problem['answer'])

In [ ]:
model_dir = Path.cwd() / 'models' / 'Qwen3.5-0.8B'
tokenizer = AutoTokenizer.from_pretrained(
    model_dir, local_files_only=True, trust_remote_code=False
)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    model_dir, dtype=torch.bfloat16, local_files_only=True,
    trust_remote_code=False, use_safetensors=True,
).to('cuda').eval()

edits = load_edit()
for edit in edits:
    actual = tensor_sha256(model.get_parameter(edit['name']))
    if actual != edit['base_sha256']:
        raise RuntimeError('The loaded base tensor differs from the published edit input.')
    print(edit['name'], '- base SHA-256 verified')
    print('  changed values:', edit['changed_entries'], 'columns:', edit['changed_columns'])
print('The blog counts 131,072 touched values; 130,188 actually differ after BF16 rounding.')

## 1. Original model

Each problem gets its own fixed seed. We grade only a simple integer in the last `\boxed{...}` **after** `</think>`; a number appearing inside unfinished reasoning is not counted as a final answer. This intentionally simple grader is not the article's math-verify grader.

In [ ]:
before = run_probes(
    model, tokenizer, problems, condition='base',
    max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
)

## 2. Exact edited tensors, then automatic restoration

The helper verifies the JSON/NPZ file hashes, original tensor hashes, sparse indices, and reconstructed tensor hashes before applying anything. The edited model exists only inside the `with` block. The original two tensors are restored and hash-checked even if generation raises an exception or is interrupted. No checkpoint file is rewritten.

In [ ]:
with edited_weights(model, layers=EDIT_LAYERS) as changes:
    for change in changes:
        print('Applied:', change['name'], '- edited SHA-256 verified')
    after = run_probes(
        model, tokenizer, problems, condition='published_edit',
        max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
    )

for edit in edits:
    assert tensor_sha256(model.get_parameter(edit['name'])) == edit['base_sha256']
print('Original weights restored: both base SHA-256 hashes match.')

In [ ]:
records = before + after
print(f"{'condition':<16} {'problem':<10} {'boxed':<8} {'correct':<8} {'tokens':<8} {'cap':<6} {'repeat4':<8}")
for row in records:
    print(f"{row['condition']:<16} {row['problem_id']:<10} {str(row['boxed_answer']):<8} "
          f"{str(row['correct']):<8} {row['generated_tokens']:<8} "
          f"{str(row['hit_token_limit']):<6} {row['repeated_4gram_fraction']:<8.1%}")
print('\nrepeat4 is a token n-gram repetition heuristic, not a validated loop detector.')
print('Compare answers and reasoning, not just token counts. This sample is too small for a benchmark claim.')

In [ ]:
for row in (before[0], after[0]):
    print('\n---', row['condition'], '/', row['problem_id'], '(first 2000 characters) ---')
    print(row['text'][:2000])

In [ ]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
output_dir = Path.cwd() / 'results' / f'published_edit_{run_id}'
output_dir.mkdir(parents=True, exist_ok=False)
report = {
    'experiment': 'short paired smoke test, not a benchmark reproduction',
    'model_revision': REVISION,
    'edit_commit': UPSTREAM_COMMIT,
    'edit_layers': list(EDIT_LAYERS),
    'sampling': SAMPLING,
    'max_new_tokens': MAX_NEW_TOKENS,
    'thinking': True,
    'torch': torch.__version__,
    'transformers': version('transformers'),
    'restored_base_hashes_verified': True,
    'records': records,
}
with (output_dir / 'comparison.json').open('x', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2)
print('Full responses and metrics saved to:', output_dir / 'comparison.json')

## Turn replay into an investigation

1. Inspect the actual before/after text. Did the edit change looping, answer correctness, or just response length?
2. Try `EDIT_LAYERS = (15,)`, then `(19,)`, rerunning the comparison with the same settings. These are ablation controls, not newly discovered improvements.
3. After checking runtime, increase the cap to 1024 or 2048 and try all four supplied problems. A too-short cap can hide an effect.
4. Before generalizing, evaluate fresh held-out problems and several seeds. Do not tune an edit on the same examples you use to claim a gain.

The model is back to its original weights when this notebook finishes. The source checkpoint stays unchanged.